In [7]:
import pandas as pd
import re

def extract_answer(text: str) -> str:
    """Extract the judgment from the model's response."""
    match = re.search(r'<answer>(.*?)</answer>', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text


In [8]:
file_name = './Data/data.csv'
data_category = pd.read_csv(file_name)
data_category.head(3)

,RowID,Model,Caption Notes,annotator,video_path,Prompt,Question,Answer,Categorization-1,Categorization-2,set
0,59,Runawaygen2,NaN,wxy,./Runawaygen2/1uPVDzHcNEAApEhzlewBnx639vlNPHhL...,Generate a video for launching two rockets,How many rockets are launched in the video?,Two,Alignment,Alignment - Entity Counting,test
1,59,Runawaygen2,NaN,wxy,./Runawaygen2/1uPVDzHcNEAApEhzlewBnx639vlNPHhL...,Generate a video for launching two rockets,Do all rockets launch at the same pace and cle...,"No, one of them is left behind and still held ...",Physics,Spatial-temporal Consistency - Temporal Dynamics,test
2,59,Runawaygen2,NaN,wxy,./Runawaygen2/1uPVDzHcNEAApEhzlewBnx639vlNPHhL...,Generate a video for launching two rockets,Are there any fast-moving objects in this video?,"Yes, there are two launching rockets.",Alignment,Spatial-temporal Consistency - Spatial Dynamics,test


In [9]:
def compute_accuracy(judgments):
    correct = sum('correct' in j.lower() and 'incorrect' not in j.lower() for j in judgments)
    total = len(judgments)
    # print('Total:', total, 'Correct:', correct)
    return round(correct / total, 4)*100 if total > 0 else 0.0

def compute_accuracy_by_category(judgments, categories):
    ### Given judgments and categories, compute accuracy for each category
    # print('Total judgments:', len(judgments))
    category_map = {}
    for j, c in zip(judgments, categories):
        if c not in category_map:
            category_map[c] = {'correct': 0, 'total': 0}
        if 'correct' in j.lower() and 'incorrect' not in j.lower():
            category_map[c]['correct'] += 1
        category_map[c]['total'] += 1
    # Compute accuracy for each category
    accuracy_by_category = {c: round(v['correct'] / v['total']*100, 2) if v['total'] > 0 else 0.0 for c, v in category_map.items()}
    return accuracy_by_category

In [29]:
Models_name = 'Qwen2.5-VL-7B-Instruct'
# Models_name = 'VideoHallu-R1-v1.0'

In [30]:
file_name = f'./eval_results/{Models_name}.csv'

eval_df = pd.read_csv(file_name)
eval_df['set'] = data_category['set']
eval_df['Categorization-1'] = data_category['Categorization-1']
eval_df['Categorization-2'] = data_category['Categorization-2']
eval_df.head(3)

,Unnamed: 0.1,Unnamed: 0,RowID,Model,Question,Answer,Question_category,Question_sub_category,Caption Notes,annotator,video_path,Prompt,Qwen2.5-VL-7B-Instruct,Qwen2.5-VL-7B-Instruct_judgment,set,Categorization-1,Categorization-2
0,0,0,59.0,Runawaygen2,How many launching rockets in this video?,Two,Alignment,NaN,NaN,wxy,./new_video_folders/Runawaygen2/1uPVDzHcNEAApE...,Generate a video for launching two rockets,There are two launching rockets in the video.,correct,test,Alignment,Alignment - Entity Counting
1,1,1,59.0,Runawaygen2,Does all the launching rockets lifting up in t...,No. One of them left behind and is still not r...,Alignment,NaN,NaN,wxy,./new_video_folders/Runawaygen2/1uPVDzHcNEAApE...,Generate a video for launching two rockets,"No, not all the launching rockets lift up at t...",correct,test,Physics,Spatial-temporal Consistency - Temporal Dynamics
2,2,2,59.0,Runawaygen2,Is there any fast moving objects inside this v...,Yes. There are two launching rockets inside.,Reasoning,NaN,NaN,wxy,./new_video_folders/Runawaygen2/1uPVDzHcNEAApE...,Generate a video for launching two rockets,"Yes, there is a fast-moving object in the vide...",correct,test,Alignment,Spatial-temporal Consistency - Spatial Dynamics


In [31]:
mask = eval_df["set"] == "test"
subset = eval_df.loc[mask]
len(subset)

1525

In [32]:
compute_accuracy(subset[f'{Models_name}_judgment'])

54.16

In [33]:
print('Computing Super-category accuracy:')
compute_accuracy_by_category(subset[f'{Models_name}_judgment'], subset['Categorization-1'])

Computing Super-category accuracy:


{'Alignment': 59.6,
 'Physics': 45.57,
 'Spatial-temporal consistency': 48.59,
 'Common sense reasoning': 43.75}

In [34]:
print('Computing Sub-category accuracy:')
compute_accuracy_by_category(subset[f'{Models_name}_judgment'], subset['Categorization-2'])

Computing Sub-category accuracy:


{'Alignment - Entity Counting': 60.54,
 'Spatial-temporal Consistency - Temporal Dynamics': 46.91,
 'Spatial-temporal Consistency - Spatial Dynamics': 43.66,
 'Alignment - Entity Properties': 61.08,
 'Physics - Motion': 51.11,
 'Physics - State Transition': 40.0,
 'Alignment - Entity Recognition and Classification': 67.82,
 'Spatial-temporal Consistency - Camera Dynamics': 59.52,
 'Common Sense - Knowledge': 55.74,
 'Physics - Constraints and Properties': 58.82,
 'Alignment - Spatial Relationships': 57.14,
 'Common Sense - Reasoning': 55.07,
 'Physics - Conservation': 42.86}